<a href="https://colab.research.google.com/github/joseguilhermemarinho/big_data_proj/blob/main/codigo/ingestao_av1_big_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import requests
import pandas as pd
from datetime import datetime
import pytz
import os

url = "https://dados.mobilidade.rio/gps/sppo"
fuso_horario = pytz.timezone('America/Recife')

In [21]:
response = requests.get(url)

if response.status_code == 200:
  dados_brutos = response.json()
  df_gps = pd.DataFrame(dados_brutos)

  df_gps['momento_extracao'] = datetime.now(fuso_horario).strftime('%Y-%m-%d %H:%M:%S')

  nome_arquivo = "gps_sppo_bruto.csv"
  df_gps.to_csv(nome_arquivo, index=False)

  print(f"Foram ingeridos {len(df_gps)} registros de GPS.")
  display(df_gps.head())

else:
  print(f"Erro ao acessar a API. Status: {response.status_code}")

Foram ingeridos 508468 registros de GPS.


,ordem,latitude,longitude,datahora,velocidade,linha,datahoraenvio,datahoraservidor,momento_extracao
0,D33207,"-22,90246","-43,56223",1780399285000,26,822,1780399297000,1780399302000,2026-06-02 09:21:55
1,D33318,"-22,85487","-43,39731",1780399287000,39,764,1780399297000,1780399302000,2026-06-02 09:21:55
2,D33307,"-22,87964","-43,46424",1780399287000,24,741,1780399297000,1780399302000,2026-06-02 09:21:55
3,D33172,"-22,87953","-43,45609",1780399288000,22,746,1780399297000,1780399302000,2026-06-02 09:21:55
4,D33177,"-22,88611","-43,61785",1780399287000,33,756,1780399297000,1780399302000,2026-06-02 09:21:55


aacessa os dados via API, acrescenta o momento do salvamento e cria o primeiro arquivo bruto. porém, o arquivo bruto é muito grande, precisamos garantir que não vai dar trabalho à equipe de transformação.

In [22]:
tamanho_mb = os.path.getsize("gps_sppo_bruto.csv") / (1024 * 1024)
print(f"O arquivo original tem: {tamanho_mb:.2f} MB")

nome_amostra_github = "amostra_gps_sppo_bruto.csv"
df_gps.head(2000).to_csv(nome_amostra_github, index=False)

print(f"Amostra reduzida com sucesso")

O arquivo original tem: 48.84 MB
Amostra reduzida com sucesso


a extração completa resultou em quase meio milhão de registros, o que gera um arquivo muito pesado. a redução para 2000 linhas foi aplicada para cumprir estritamente as diretrizes do projeto, que determinam que a pasta /dados deve conter apenas amostras pequenas e que arquivos grandes não devem ser commitados. isso garante a performance do repositório no GitHub e evita bloqueios de tamanho.

In [23]:
df = df_gps.copy()

colunas_timestamp = ['datahora', 'datahoraenvio', 'datahoraservidor']

for col in colunas_timestamp:
  df[col] = pd.to_datetime(pd.to_numeric(df[col], errors='coerce'), unit='ms', utc=True)
  df[col] = df[col].dt.tz_convert('America/Sao_Paulo')

print("Exemplo após conversão:")
print(df[['datahora', 'datahoraenvio', 'datahoraservidor']].head())

Exemplo após conversão:
                   datahora             datahoraenvio  \
0 2026-06-02 08:21:25-03:00 2026-06-02 08:21:37-03:00   
1 2026-06-02 08:21:27-03:00 2026-06-02 08:21:37-03:00   
2 2026-06-02 08:21:27-03:00 2026-06-02 08:21:37-03:00   
3 2026-06-02 08:21:28-03:00 2026-06-02 08:21:37-03:00   
4 2026-06-02 08:21:27-03:00 2026-06-02 08:21:37-03:00   

           datahoraservidor  
0 2026-06-02 08:21:42-03:00  
1 2026-06-02 08:21:42-03:00  
2 2026-06-02 08:21:42-03:00  
3 2026-06-02 08:21:42-03:00  
4 2026-06-02 08:21:42-03:00  


Os dados nos são enviados em tempo real. A cada vez que executamos a célula que puxa os dados da api, são enviados novos dados para nós.

As colunas de datahora, datahoraenvio e datahoraservidor estão todas em um timestamp unix normalmente utilizados em sistema iot, o que acontece com os dados desse datalake.

Por isso, a conversão desse timestamp para o formato de yyyy-mm-dd hh:mm:ss - GMT-3 é muito importante para que nós possamos visualizar melhor e também entendermos perfeitamente a "tradução" do timestamp.

In [24]:
#Criando coluinas derivadas a partir do datahora

df['data'] = df['datahora'].dt.date
df['hora'] = df['datahora'].dt.hour
df['minuto'] = df['datahora'].dt.minute
df['dia_semana'] = df['datahora'].dt.day_name()

#Calculando latencia de envio: tempo entre captura do GPS e envio (ms)
df['latencia_envio_s'] = (
    df['datahoraenvio'] - df['datahora']
).dt.total_seconds()

#Calculando agora latencia do servidor: tempo entre envio e chegada ao servidor (ms)
df['latencia_servidor_s'] = (
    df['datahoraservidor'] - df['datahoraenvio']
).dt.total_seconds()

print("\nLatências calculadas:")
print(df[['latencia_envio_s', 'latencia_servidor_s']].describe())


Latências calculadas:
       latencia_envio_s  latencia_servidor_s
count     508468.000000        508468.000000
mean          20.560944            16.231527
std          360.740740             8.932626
min            0.000000           -16.000000
25%            5.000000             9.000000
50%            8.000000            16.000000
75%           10.000000            24.000000
max        32568.000000            49.000000


In [25]:
print("\nNulos antes da limpeza:")
print(df.isnull().sum())


Nulos antes da limpeza:
ordem                  0
latitude               0
longitude              0
datahora               0
velocidade             0
linha                  0
datahoraenvio          0
datahoraservidor       0
momento_extracao       0
data                   0
hora                   0
minuto                 0
dia_semana             0
latencia_envio_s       0
latencia_servidor_s    0
dtype: int64


Por mais que não existam números nulos antes da limpeza, é importante lembrar que os dados que estamos tratando são todos em tempo real. A cada nova execução de linha de código, novos dados chegam.

Por isso, precisamos normalizar, padronizar e estruturar corretamente cada dado novo que chega pois isso facilitará e muito na nossa análise da camada gold.

In [26]:
# Substituir vírgula por ponto ANTES de converter para numérico
for col in ['latitude', 'longitude', 'velocidade']:
    df[col] = df[col].astype(str).str.replace(',', '.', regex=False)
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop de linhas com nulos em colunas críticas
colunas_criticas = ['ordem', 'latitude', 'longitude', 'datahora', 'velocidade']
df = df.dropna(subset=colunas_criticas)

# Remover coordenadas fora do Brasil
df = df[
    (df['latitude'].between(-34, 5)) &
    (df['longitude'].between(-74, -28))
]

# Remover velocidades negativas
df = df[df['velocidade'] >= 0]

print("Shape após limpeza:", df.shape)
print("\nExemplo de coordenadas após correção:")
print(df[['ordem', 'latitude', 'longitude', 'velocidade']].head())

Shape após limpeza: (508468, 15)

Exemplo de coordenadas após correção:
    ordem  latitude  longitude  velocidade
0  D33207 -22.90246  -43.56223          26
1  D33318 -22.85487  -43.39731          39
2  D33307 -22.87964  -43.46424          24
3  D33172 -22.87953  -43.45609          22
4  D33177 -22.88611  -43.61785          33


In [27]:
# Salvando na camada silver em um diretório local válido
os.makedirs('./silver', exist_ok=True)  # Cria o diretório 'silver' na pasta atual

df['data'] = pd.to_datetime(df['data'], errors='coerce')  # Converte a coluna 'data' para datetime

# Salva os arquivos no diretório 'silver'
df.to_parquet('./silver/sppo_silver.parquet', index=False)
df.to_csv('./silver/sppo_silver.csv', index=False)

# Cria uma amostra e salva
df_amostra = df.sample(n=1000, random_state=42)
df_amostra.to_csv('./silver/sppo_amostra.csv', index=False)

print("Salvo em /content/silver/")
print("Shape final:", df.shape)

Salvo em /content/silver/
Shape final: (508468, 15)


In [28]:
df.head()

,ordem,latitude,longitude,datahora,velocidade,linha,datahoraenvio,datahoraservidor,momento_extracao,data,hora,minuto,dia_semana,latencia_envio_s,latencia_servidor_s
0,D33207,-22.90246,-43.56223,2026-06-02 08:21:25-03:00,26,822,2026-06-02 08:21:37-03:00,2026-06-02 08:21:42-03:00,2026-06-02 09:21:55,2026-06-02,8,21,Tuesday,12.0,5.0
1,D33318,-22.85487,-43.39731,2026-06-02 08:21:27-03:00,39,764,2026-06-02 08:21:37-03:00,2026-06-02 08:21:42-03:00,2026-06-02 09:21:55,2026-06-02,8,21,Tuesday,10.0,5.0
2,D33307,-22.87964,-43.46424,2026-06-02 08:21:27-03:00,24,741,2026-06-02 08:21:37-03:00,2026-06-02 08:21:42-03:00,2026-06-02 09:21:55,2026-06-02,8,21,Tuesday,10.0,5.0
3,D33172,-22.87953,-43.45609,2026-06-02 08:21:28-03:00,22,746,2026-06-02 08:21:37-03:00,2026-06-02 08:21:42-03:00,2026-06-02 09:21:55,2026-06-02,8,21,Tuesday,9.0,5.0
4,D33177,-22.88611,-43.61785,2026-06-02 08:21:27-03:00,33,756,2026-06-02 08:21:37-03:00,2026-06-02 08:21:42-03:00,2026-06-02 09:21:55,2026-06-02,8,21,Tuesday,10.0,5.0


Com base no fato de que temos dados que não sempre atualizados em tempo real, termos organizado a questão do timestamp, termos latitude e longitude e conseguirmos criar novas colunas com dados que podem nos ajudar em análises mais complexas, percebemos alguns insights:



1.   Podemos monitorar pontos de trânsito dentro da cidade, inclusive conseguindo apontar o ponto específico por conta da latitude e longitude.
2.   Podemos também monitorar linhas de transporte público, identificando mudanças que poderiam acontecer, como remoção ou adição de novas linhas.
3. Podemos criar gráficos para identificar horários de pico e velocidade média de cada local, gerando insights de melhoria de infraestrutura da cidade e mobilidade urbana.



In [29]:
import folium
from math import radians, sin, cos, sqrt, atan2

center_lat = df['latitude'].mean()
center_lon = df['longitude'].mean()

def haversine(lat1, lon1, lat2, lon2):
  R = 6371.0
  dlat = radians(lat2 - lat1)
  dlon = radians(lon2 - lon1)

  a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
  c = 2 * atan2(sqrt(a), sqrt(1-a))

  return R * c

distances = df.apply(lambda row: haversine(center_lat, center_lon, row['latitude'], row['longitude']), axis=1)
radius = distances.max()

# 4. Criar mapa
m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

# 5. Adicionar pontos
for _, row in df.iterrows():
    folium.Marker([row['latitude'], row['longitude']]).add_to(m)

# 6. Adicionar círculo
folium.Circle(
    location=[center_lat, center_lon],
    radius=radius,
    color='blue',
    fill=True,
    fill_opacity=0.2
).add_to(m)


df_sample = df.sample(5000)  # pega só 5k pontos

points = df_sample[['latitude', 'longitude']].values.tolist()

m = folium.Map(location=[df['latitude'].mean(), df['longitude'].mean()], zoom_start=12)

from folium.plugins import MarkerCluster
MarkerCluster(points).add_to(m)


In [30]:
m